In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import datasets
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter

import numpy as np

# LeNet-5 구현

## 데이터 불러오기 및 전처리

In [39]:
data_transform = transforms.Compose(
    [
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize((0.5,),(1.0,))             # 간이 정규화 (원랜 이렇게 하면 좋은건 아님)
    ]
)

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=data_transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=data_transform)

train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               Resize(size=32, interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=(0.5,), std=(1.0,))
           )

In [40]:
train_data.data.shape

torch.Size([60000, 28, 28])

In [41]:
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

In [42]:
next(iter(train_loader))[0].shape

torch.Size([128, 1, 32, 32])

## 모델 클래스 정의부

In [43]:
class ScaledTanh(nn.Module):
    def __init__(self, alpha=1.7159, beta=2.0/3.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, x):
        return self.alpha * torch.tanh(self.beta * x)



class LearnableSubsampling(nn.Module):
    def __init__(self, channels, pool_kernel=2, pool_stride=2):
        super().__init__()
        self.pool = nn.AvgPool2d(kernel_size=pool_kernel, stride=pool_stride)
        # one scale and bias per channel
        self.a = nn.Parameter(torch.ones(channels, 1, 1))
        self.b = nn.Parameter(torch.zeros(channels, 1, 1))
        self.act = ScaledTanh()

    def forward(self, x):
        # x: (N, C, H, W)
        y = self.pool(x)
        y = y * self.a + self.b
        return self.act(y)


class PartialConvC3(nn.Module):
    """
    Input:  (N, 6, 14, 14)  from S2
    Output: (N,16, 10, 10)  after 5x5 conv valid
    """
    def __init__(self, in_channels=6, out_channels=16, kernel_size=5):
        super().__init__()
        assert in_channels == 6 and out_channels == 16, "LeNet-5 C3 expects 6->16."

        # Classic LeNet-5 C3 connection scheme (16 maps)
        self.connections = [
            [0, 1, 2],
            [1, 2, 3],
            [2, 3, 4],
            [3, 4, 5],
            [0, 4, 5],
            [0, 1, 5],
            [0, 1, 2, 3],
            [1, 2, 3, 4],
            [2, 3, 4, 5],
            [0, 3, 4, 5],
            [0, 1, 4, 5],
            [0, 1, 2, 5],
            [0, 1, 3, 4],
            [1, 2, 4, 5],
            [0, 2, 3, 5],
            [0, 1, 2, 3, 4, 5],
        ]

        self.convs = nn.ModuleList([
            nn.Conv2d(len(conn), 1, kernel_size=kernel_size, stride=1, padding=0, bias=True)
            for conn in self.connections
        ])
        self.act = ScaledTanh()

    def forward(self, x):
        # x: (N, 6, H, W)
        outs = []
        for conv, conn in zip(self.convs, self.connections):
            xi = x[:, conn, :, :]          # (N, len(conn), H, W)
            yi = conv(xi)                  # (N, 1, H-4, W-4)
            outs.append(yi)
        y = torch.cat(outs, dim=1)         # (N,16, H-4, W-4)
        return self.act(y)



class LeNet5(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.act = ScaledTanh()
        
        self.c1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1)
        self.s2 = LearnableSubsampling(6, pool_kernel=2, pool_stride=2)

        self.c3 = PartialConvC3(in_channels=6, out_channels=16, kernel_size=5)
        self.s4 = LearnableSubsampling(16, pool_kernel=2, pool_stride=2)

        self.c5 = nn.Conv2d(16, 120, kernel_size=5, stride=1, padding=0, bias=True)
        self.f6 = nn.Linear(120, 84, bias=True)
        self.out = nn.Linear(84, 10, bias=True)

        self._init_like_lenet()        
        
        
    def _init_like_lenet(self):
        # The paper predates modern init defaults; this is a reasonable classic choice.
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.uniform_(m.weight, -0.1, 0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        # subsampling params start near identity
        nn.init.ones_(self.s2.a)
        nn.init.zeros_(self.s2.b)
        nn.init.ones_(self.s4.a)
        nn.init.zeros_(self.s4.b)
    
    
    def forward(self, x):
        x = self.act(self.c1(x))   # C1
        x = self.s2(x)             # S2
        x = self.c3(x)             # C3
        x = self.s4(x)             # S4
        x = self.act(self.c5(x))   # C5 -> (N,120,1,1)
        x = x.view(x.size(0), -1)  # (N,120)
        x = self.act(self.f6(x))   # F6
        x = self.out(x)            # OUT (linear)
        return x


model = LeNet5()
model

LeNet5(
  (act): ScaledTanh()
  (c1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (s2): LearnableSubsampling(
    (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (act): ScaledTanh()
  )
  (c3): PartialConvC3(
    (convs): ModuleList(
      (0-5): 6 x Conv2d(3, 1, kernel_size=(5, 5), stride=(1, 1))
      (6-14): 9 x Conv2d(4, 1, kernel_size=(5, 5), stride=(1, 1))
      (15): Conv2d(6, 1, kernel_size=(5, 5), stride=(1, 1))
    )
    (act): ScaledTanh()
  )
  (s4): LearnableSubsampling(
    (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (act): ScaledTanh()
  )
  (c5): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (f6): Linear(in_features=120, out_features=84, bias=True)
  (out): Linear(in_features=84, out_features=10, bias=True)
)

In [53]:
num_epochs = 10
lr = 1e-3
momentum = 0.9

In [54]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LeNet5(
  (act): ScaledTanh()
  (c1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (s2): LearnableSubsampling(
    (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (act): ScaledTanh()
  )
  (c3): PartialConvC3(
    (convs): ModuleList(
      (0-5): 6 x Conv2d(3, 1, kernel_size=(5, 5), stride=(1, 1))
      (6-14): 9 x Conv2d(4, 1, kernel_size=(5, 5), stride=(1, 1))
      (15): Conv2d(6, 1, kernel_size=(5, 5), stride=(1, 1))
    )
    (act): ScaledTanh()
  )
  (s4): LearnableSubsampling(
    (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (act): ScaledTanh()
  )
  (c5): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (f6): Linear(in_features=120, out_features=84, bias=True)
  (out): Linear(in_features=84, out_features=10, bias=True)
)

In [47]:
summary(model, input_size=(1, 64, 64))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x9720 and 120x84)

In [60]:
writer = SummaryWriter()

count = 0

# checkpoint 불러오기

for ep in range(num_epochs):
    
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, leave=True)
    for X, y in loop:
        X, y = X.to("cuda"), y.to("cuda")
        optimizer.zero_grad()
        logits = model(X)
        
        # mse를 쓰기 위해 바꿔줘야함..
        #target = torch.full_like(logits, -1.0)
        #target[torch.arange(y.size(0), device=y.device), y] = 1.0
        
        loss = criterion(logits, y)
        writer.add_scalar("Loss/train", loss.item(), count)
        count += 1
        running_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        loop.set_description(f"Epoch [{ep+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(train_loader)
    print(f"\nEpoch {ep+1} 평균 Loss: {epoch_loss:.4f}")

Epoch [1/10]: 100%|██████████| 469/469 [00:18<00:00, 25.88it/s, loss=0.0266] 



Epoch 1 평균 Loss: 0.0421


Epoch [2/10]: 100%|██████████| 469/469 [00:18<00:00, 25.87it/s, loss=0.065]  



Epoch 2 평균 Loss: 0.0410


Epoch [3/10]: 100%|██████████| 469/469 [00:17<00:00, 26.17it/s, loss=0.012]  


...


Epoch 8 평균 Loss: 0.0357


Epoch [9/10]: 100%|██████████| 469/469 [00:17<00:00, 26.61it/s, loss=0.125]  



Epoch 9 평균 Loss: 0.0351


Epoch [10/10]: 100%|██████████| 469/469 [00:17<00:00, 27.48it/s, loss=0.0122] 


Epoch 10 평균 Loss: 0.0344


In [56]:
# due to deprecation of setuptools, you need to downgrade its version <82.0.0 then run, for example:
!uv pip install setuptools==80.10.2
!tensorboard --logdir=runs

Audited 1 package in 47ms


^C


In [61]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for img, y in test_loader:
        img, y = img.to(device), y.to(device)
        
        logits = model(img)
        _, y_pred = torch.max(logits, dim=1)
        
        correct += (y_pred==y).sum().item()
        total += y.size(0)

print(f"test acc: {correct/total}")

test acc: 0.9861


In [101]:
F.softmax(logits)

C:\Users\user\AppData\Local\Temp\ipykernel_14156\383899727.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  F.softmax(logits)


tensor([[0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509, 0.0610,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.4509,
         0.0610],
        [0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.4509],
        [0.4509, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610, 0.0610,
         0.0610],
        [0

# 학습된 모델 저장방법들

## 방법1 - 권장

In [62]:
# model 가중치만 저장하는 방법 1 - 가장 권장되고 많이 쓰임.
torch.save(model.state_dict(), "./models/LeNet_CNN_params.pth")

In [63]:
model_new = LeNet5()
model_new.load_state_dict(torch.load("./models/LeNet_CNN_params.pth"))

<All keys matched successfully>

## 방법2 - 보안에 취약함 (deprecated)
* 모델은 저장되는 순간 구조를 알 수 없거나 거의 불가능. 유추는 가능

In [64]:
# 모델 저장 방법 2
# 모델구조+가중치 통째로 저장하는 방법.
## 바로 불러올 수 없음. 모델 로드 파일에서 설명 cont.
## summary()를 실행한 뒤 저장이 안됨.
torch.save(model_new,"./models/LeNet_CNN_all.pth")

## 방법3 - 가장 권장

In [65]:
# 모델 저장 방법 3
# checkpoint 활용방법
torch.save({
    "epoch": ep,
    "model_state_dict": model.state_dict(),
    "model_config": {
        "lr": lr,
        "num_epochs": num_epochs,
        "batch_size": 128
    },
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss
}, "./models/checkpoint.pth")

checkpoint = torch.load("./models/checkpoint.pth")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
epoch = checkpoint["epoch"]
loss = checkpoint["loss"]